# 03 — Data Correction Checks

This notebook tests **whether all the data-correction transformations are conducted** and **whether the validation tests are working**.

It runs the corrections from `src/correct_data_issues.py` on the first-transaction churn table, then runs every validation check from `src/data_validation_tests.py` to confirm none of the 7 data-quality issues (defined in `02_data_quality_first_txn.ipynb`) survive. If any correction is incomplete, `validate_all` raises a `DataValidationError` listing every check that did not pass.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make the src/ modules importable from the notebooks/ directory.
SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from correct_data_issues import IN_FILE, correct_data_issues
from data_validation_tests import validate_all, CHECKS

## 1. Run the transformations

Load the first-transaction table (`data/processed/first_transaction_churn.csv`) and apply `correct_data_issues`. The before/after shapes confirm the corrections actually ran.

In [2]:
raw = pd.read_csv(IN_FILE, parse_dates=['invoice_date'])
clean = correct_data_issues(raw)

print(f'Before corrections : {raw.shape}   ({raw["customer_id"].nunique():,} customers)')
print(f'After corrections  : {clean.shape}   ({clean["customer_id"].nunique():,} customers)')
print(f'Rows removed       : {len(raw) - len(clean):,}')

Before corrections : (126080, 10)   (5,346 customers)
After corrections  : (125043, 10)   (5,045 customers)
Rows removed       : 1,037


## 2. Validate the corrections

Run every validation check on the corrected data. `validate_all` raises a `DataValidationError` (listing all failing checks) if any issue remains; if it returns without raising, all corrections are confirmed completed.

In [3]:
validate_all(clean)
print(f'All {len(CHECKS)} validation checks passed on {len(clean):,} rows '
      f'({clean["customer_id"].nunique():,} customers) — no known issues remain.')

All 4 validation checks passed on 125,043 rows (5,045 customers) — no known issues remain.
